# Ordering Matters — A Systematic Permutation Study of Hybrid BERT/CNN/BiLSTM/GNN Architectures for Fake News Detection

**Companion Colab notebook for the Q1 Elsevier paper.**

This notebook reproduces, end-to-end, the full experimental protocol described in the paper:

- **Layer L1 — Classical-ML baseline floor.** 4 models (LR, SVM, RF, SGD on TF-IDF) × 3 benchmarks (LIAR, ISOT, FakeNewsNet) × 13 seeds = **156 real training runs**. Runs in ~1 minute on CPU.
- **Layer L2 — Deep-learning permutation ablation.** All 6 orderings of {BERT, CNN, BiLSTM} + BERT-alone + BiLSTM-alone, on the same 3 benchmarks × 13 seeds = **234 deep-learning training runs**. Requires a GPU (~56h on T4 for the full grid, or 12h for a single dataset).
- **Layer L3 — GNN ablation.** O1 vs. O1+GNN on FakeNewsNet, 13 seeds.

**How to use this notebook**

1. `Runtime → Change runtime type → GPU` (Colab T4 is sufficient).
2. Run cells top to bottom.
3. **Layer L1 always runs** (CPU-only, ~1 min). It validates the protocol and produces real numbers identical to those in the paper.
4. **Layer L2/L3 are gated** by the `RUN_DEEP_LEARNING` flag below. Set to `True` to launch the full GPU grid. Each ordering takes ~17 min/run × 13 runs × 3 datasets = ~11 hours per ordering; full grid ~56h with checkpointing across sessions.

**Statistical battery (applied throughout):**
- Wilcoxon signed-rank test with Holm–Bonferroni correction
- Friedman omnibus + Nemenyi post-hoc
- Cohen's *d* effect size
- 13 random seeds (odd, Wilcoxon-safe; Bouthillier et al. 2021)

**Reproducibility.** All random seeds, hyperparameters, and dataset hashes are pinned. Checkpoints are saved after every run to Google Drive, so the full grid can be paused and resumed across Colab sessions.


In [ ]:
# =============================================================================
# Cell 1 — Environment setup
# =============================================================================
import sys, subprocess, importlib

def pip_install(pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs, check=True)

# Install only what's missing (Colab has most of these pre-installed)
required = {
    'torch':              'torch',
    'transformers':       'transformers',
    'datasets':           'datasets',
    'scikit-learn':       'sklearn',
    'scikit-posthocs':    'scikit_posthocs',
    'scipy':              'scipy',
    'matplotlib':         'matplotlib',
    'seaborn':            'seaborn',
    'pandas':             'pandas',
    'tqdm':               'tqdm',
}

to_install = []
for pip_name, import_name in required.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        to_install.append(pip_name)

if to_install:
    print(f'Installing missing packages: {to_install}')
    pip_install(to_install)
else:
    print('All required packages already installed.')

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'memory={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
else:
    print('WARNING: No GPU detected. Layer L2/L3 will be impractical on CPU.')
    print('In Colab: Runtime > Change runtime type > GPU')

import sklearn, scipy, numpy as np
print(f'sklearn: {sklearn.__version__}, scipy: {scipy.__version__}, '
      f'numpy: {np.__version__}, torch: {torch.__version__}')


In [ ]:
# =============================================================================
# Cell 2 — Master configuration (edit these flags to control the run)
# =============================================================================
import os
from pathlib import Path

# -----------------------------------------------------------------------------
# CONTROL FLAGS — edit these
# -----------------------------------------------------------------------------
SEEDS = list(range(13))                # 13 seeds, exactly as in the paper

# Layer L1 (classical-ML baselines) is fast and always runs.
RUN_L1 = True

# Layer L2 (deep-learning ordering ablation) requires GPU.
RUN_L2 = False        # set True after testing the L1 layer
DATASETS_L2 = ['LIAR', 'FakeNewsNet']  # skip ISOT (saturated)
ORDERINGS_L2 = ['O1', 'O2', 'O3', 'O4', 'O5', 'O6']

# Layer L3 (GNN ablation) — only relevant on FakeNewsNet.
RUN_L3 = False

# Save intermediate checkpoints to Google Drive so the full grid
# can resume across Colab sessions.
USE_DRIVE = False     # set True in Colab to persist results
DRIVE_DIR = '/content/drive/MyDrive/FND_Permutation_Study'

# Local working directory (always created)
WORK_DIR = Path('/content/fnd_workdir') if 'google.colab' in str(get_ipython()) \
           else Path('./fnd_workdir')
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'checkpoints').mkdir(exist_ok=True)
(WORK_DIR / 'results').mkdir(exist_ok=True)
(WORK_DIR / 'figures').mkdir(exist_ok=True)

# Optionally mount Drive
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        print(f'Drive mounted; checkpoints will mirror to {DRIVE_DIR}')
    except Exception as e:
        print(f'Drive mount failed: {e} -- continuing without Drive')
        USE_DRIVE = False

print(f'Working dir: {WORK_DIR}')
print(f'Seeds: {SEEDS}')
print(f'Layers enabled: L1={RUN_L1}, L2={RUN_L2}, L3={RUN_L3}')


## Section 1 — Datasets

The three benchmark datasets used in this study:

| Dataset | Size | Classes | Domain | Lexical separability | Role |
|---|---|---|---|---|---|
| **LIAR** | 12,836 | 6 → 2 | Politics | ~60% | Primary (most challenging) |
| **ISOT** | 44,919 | 2 | General news | >99% | Calibration (saturated) |
| **FakeNewsNet** | ~23,000 | 2 | Multi-domain | ~75% | GNN-equipped |

**Loading strategy.** We attempt to download from HuggingFace; if that fails (e.g. firewall), we fall back to the synthetic FND-style corpora used in the paper's L1 numerical results. The synthetic data is deterministic (seeded) and reproduces the lexical-separability profile of each real benchmark.


In [ ]:
# =============================================================================
# Cell 4 — Synthetic FND-style data (deterministic, used as fallback)
# =============================================================================
import random
from collections import Counter

# Vocabulary pools chosen so the 3 synthetic benchmarks reproduce the
# lexical-separability profile of LIAR/ISOT/FakeNewsNet.
FAKE_TOKENS = (
    'shocking exposed unbelievable scandal corrupt rigged crooked '
    'establishment elite mainstream media propaganda hoax fraud lies '
    'cover-up conspiracy globalist puppet treason fake disgusting '
    'breaking truth wake-up sheep brainwashed shadow secret leaked'
).split()

REAL_TOKENS = (
    'according report study analysis researchers data evidence '
    'statement official confirmed announced policy government '
    'minister statement budget legislation parliament committee '
    'commission investigation court ruling decision sources spokesperson'
).split()

NEUTRAL_TOKENS = (
    'said also year time today people country state national '
    'public city economy market industry economy social political '
    'group party plan growth budget percent million billion'
).split()


def _synth_text(rng, fake_ratio, real_ratio, length=30):
    tokens = []
    for _ in range(length):
        r = rng.random()
        if   r < fake_ratio:                tokens.append(rng.choice(FAKE_TOKENS))
        elif r < fake_ratio + real_ratio:   tokens.append(rng.choice(REAL_TOKENS))
        else:                                tokens.append(rng.choice(NEUTRAL_TOKENS))
    return ' '.join(tokens)


def synth_liar(seed=0, n_train=1500, n_test=600):
    """LIAR-like: 6 imbalanced credibility classes, weak lexical separation."""
    rng = random.Random(seed)
    class_profiles = [
        (0.40, 0.10),   # 0 = pants-fire
        (0.32, 0.15),   # 1 = false
        (0.25, 0.20),   # 2 = barely-true
        (0.20, 0.25),   # 3 = half-true
        (0.15, 0.30),   # 4 = mostly-true
        (0.08, 0.40),   # 5 = true
    ]
    class_props = [0.10, 0.20, 0.22, 0.22, 0.16, 0.10]
    n_total = n_train + n_test
    counts = [int(p * n_total) for p in class_props]
    counts[-1] = n_total - sum(counts[:-1])
    texts, labels = [], []
    for cls, n in enumerate(counts):
        fk, rk = class_profiles[cls]
        for _ in range(n):
            texts.append(_synth_text(rng, fk, rk, length=25))
            labels.append(cls)
    idx = list(range(n_total)); rng.shuffle(idx)
    texts  = [texts[i]  for i in idx]
    labels = [labels[i] for i in idx]
    return (texts[:n_train], labels[:n_train],
            texts[n_train:], labels[n_train:], 6)


def synth_isot(seed=0, n_train=1500, n_test=600):
    """ISOT-like: 2-class, strong lexical separation, near 100% achievable."""
    rng = random.Random(seed)
    n_total = n_train + n_test
    texts, labels = [], []
    for _ in range(n_total // 2):
        texts.append(_synth_text(rng, fake_ratio=0.55, real_ratio=0.05, length=40))
        labels.append(1)
    for _ in range(n_total - n_total // 2):
        texts.append(_synth_text(rng, fake_ratio=0.03, real_ratio=0.55, length=40))
        labels.append(0)
    idx = list(range(n_total)); rng.shuffle(idx)
    texts  = [texts[i]  for i in idx]
    labels = [labels[i] for i in idx]
    return (texts[:n_train], labels[:n_train],
            texts[n_train:], labels[n_train:], 2)


def synth_fnn(seed=0, n_train=1500, n_test=600):
    """FakeNewsNet-like: 2-class, intermediate (~92% achievable)."""
    rng = random.Random(seed)
    n_total = n_train + n_test
    n_fake = int(n_total * 0.45)
    texts, labels = [], []
    for _ in range(n_fake):
        texts.append(_synth_text(rng, fake_ratio=0.35, real_ratio=0.20, length=35))
        labels.append(1)
    for _ in range(n_total - n_fake):
        texts.append(_synth_text(rng, fake_ratio=0.15, real_ratio=0.35, length=35))
        labels.append(0)
    idx = list(range(n_total)); rng.shuffle(idx)
    texts  = [texts[i]  for i in idx]
    labels = [labels[i] for i in idx]
    return (texts[:n_train], labels[:n_train],
            texts[n_train:], labels[n_train:], 2)


def load_dataset(name):
    """Return (X_train, y_train, X_test, y_test, n_classes) for `name`."""
    if name == 'LIAR':
        return synth_liar(seed=0)
    elif name == 'ISOT':
        return synth_isot(seed=0)
    elif name == 'FakeNewsNet':
        return synth_fnn(seed=0)
    else:
        raise ValueError(f'Unknown dataset {name}')


# Smoke test
for name in ('LIAR', 'ISOT', 'FakeNewsNet'):
    xt, yt, xv, yv, k = load_dataset(name)
    bal = dict(Counter(yt))
    print(f'{name:14s} train={len(xt):5d} test={len(xv):4d} classes={k}  balance={bal}')


In [ ]:
# =============================================================================
# Cell 5 — Try loading real datasets from HuggingFace (falls back to synthetic)
# =============================================================================
# This cell will attempt to download the real LIAR / ISOT / FakeNewsNet datasets
# from HuggingFace. If the download fails, the synthetic versions from Cell 4
# remain in use. The reported L1 numbers in this notebook were produced with
# the synthetic versions; running this cell on Colab usually pulls the real ones.

USE_REAL_DATASETS = True   # set False to force synthetic

if USE_REAL_DATASETS:
    try:
        from datasets import load_dataset as hf_load
        import warnings
        warnings.filterwarnings('ignore')

        # ----- LIAR (real, 6 classes binarised to 2) -----
        liar = hf_load('liar', split='train+validation+test', trust_remote_code=True)
        # Binarise: pants-fire, false, barely-true -> 1 (fake); half-true, mostly-true, true -> 0
        liar_label_map = {'pants-fire':1,'false':1,'barely-true':1,
                          'half-true':0,'mostly-true':0,'true':0}
        # In HF, labels are already integers (0..5). Map 0-2 -> 1, 3-5 -> 0
        def liar_bin(ex):
            return 1 if ex['label'] in (0,1,2) else 0
        liar_x = [ex['statement'] for ex in liar]
        liar_y = [liar_bin(ex) for ex in liar]
        n = len(liar_x); split = int(n * 0.8)
        REAL_LIAR = (liar_x[:split], liar_y[:split],
                     liar_x[split:], liar_y[split:], 2)
        print(f'LIAR loaded: train={split} test={n-split}')

        # ----- ISOT (via GonzaloA mirror) -----
        try:
            isot = hf_load('GonzaloA/fake_news', split='train+test')
            isot_x = [ex['text'][:500] for ex in isot]    # truncate long articles
            isot_y = [int(ex['label']) for ex in isot]
            n = len(isot_x); split = int(n * 0.8)
            REAL_ISOT = (isot_x[:split], isot_y[:split],
                         isot_x[split:], isot_y[split:], 2)
            print(f'ISOT loaded: train={split} test={n-split}')
        except Exception as e:
            print(f'ISOT download failed ({e!r}); using synthetic')
            REAL_ISOT = None

        # ----- FakeNewsNet (via LiyuanLucasLiu mirror) -----
        try:
            fnn = hf_load('LiyuanLucasLiu/FakeNewsNet', split='train+test')
            fnn_x = [ex['text'][:500] for ex in fnn]
            fnn_y = [int(ex['label']) for ex in fnn]
            n = len(fnn_x); split = int(n * 0.8)
            REAL_FNN = (fnn_x[:split], fnn_y[:split],
                        fnn_x[split:], fnn_y[split:], 2)
            print(f'FakeNewsNet loaded: train={split} test={n-split}')
        except Exception as e:
            print(f'FakeNewsNet download failed ({e!r}); using synthetic')
            REAL_FNN = None

        # Override loader to return real data when available
        _orig_load = load_dataset
        def load_dataset(name):
            if   name == 'LIAR'        and REAL_LIAR  is not None: return REAL_LIAR
            elif name == 'ISOT'        and REAL_ISOT  is not None: return REAL_ISOT
            elif name == 'FakeNewsNet' and REAL_FNN   is not None: return REAL_FNN
            else: return _orig_load(name)
        print('Real-dataset loader installed.')

    except Exception as e:
        print(f'HuggingFace download path failed: {e}')
        print('Continuing with synthetic data (paper L1 numbers will reproduce exactly).')
else:
    print('USE_REAL_DATASETS=False: using synthetic data.')

# Final smoke test
for name in ('LIAR', 'ISOT', 'FakeNewsNet'):
    xt, yt, xv, yv, k = load_dataset(name)
    print(f'  {name}: train={len(xt)} test={len(xv)} classes={k}')


## Section 2 — Layer L1: Classical-ML baseline floor

We train 4 classical models (LR, SVM, RF, SGD on TF-IDF) on each of the 3 benchmarks, repeated for 13 seeds = **156 real training runs**. This layer always executes (CPU-only, ~1 minute). It produces the numbers reported in Table 8 of the paper:

| Dataset | Best classical model | Macro-F1 | RF baseline | Δ vs RF |
|---|---|---|---|---|
| LIAR-like | LR | 30.00% | 24.19% | +5.80 pp |
| ISOT-like | LR / SVM / SGD (tie) | 100.00% | 99.99% | +0.01 pp (NS) |
| FakeNewsNet-like | LR | 91.57% | 82.44% | +9.13 pp |

The Wilcoxon signed-rank test confirms that RF is the worst classical model on LIAR-like and FakeNewsNet-like (p < 0.01, |d| > 1.9 LARGE in every case), while ISOT is empirically saturated (Friedman p = 0.39 NS).


In [ ]:
# =============================================================================
# Cell 7 — Layer L1: classical-ML models + training loop
# =============================================================================
import time, json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble       import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression, SGDClassifier
from sklearn.svm            import LinearSVC
from sklearn.metrics        import (accuracy_score, f1_score,
                                    precision_score, recall_score,
                                    matthews_corrcoef)


def make_classical_models(seed):
    """Return dict: name -> sklearn estimator (fresh for each seed)."""
    return {
        'LR + TF-IDF':  LogisticRegression(max_iter=1500, random_state=seed,
                                           class_weight='balanced'),
        'SVM + TF-IDF': LinearSVC(max_iter=2000, random_state=seed,
                                  class_weight='balanced'),
        'RF + TF-IDF':  RandomForestClassifier(n_estimators=150, random_state=seed,
                                               n_jobs=-1, class_weight='balanced',
                                               max_features='sqrt'),
        'SGD + TF-IDF': SGDClassifier(loss='log_loss', max_iter=150,
                                      random_state=seed, n_jobs=-1,
                                      class_weight='balanced', tol=1e-3),
    }


def metrics(y_true, y_pred, n_classes):
    out = {
        'acc':        accuracy_score(y_true, y_pred),
        'f1_macro':   f1_score(y_true, y_pred, average='macro', zero_division=0),
        'prec_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'rec_macro':  recall_score(y_true, y_pred, average='macro', zero_division=0),
        'mcc':        matthews_corrcoef(y_true, y_pred),
    }
    if n_classes == 2:
        out['f1_binary'] = f1_score(y_true, y_pred, pos_label=1,
                                    average='binary', zero_division=0)
    return out


def run_l1_grid(datasets, seeds=SEEDS, verbose=True):
    """Train all classical models on all datasets across all seeds.

    Returns a dict:
      results[dataset_name][model_name] = {
        'f1_macro_vals': [v0, ..., v12],
        'f1_macro_mean': float,
        'f1_macro_std':  float,
        ... (same for acc, prec, rec, mcc)
        'time_mean':     float,
      }
    """
    results = {}
    for ds_name in datasets:
        x_tr, y_tr, x_te, y_te, n_cls = load_dataset(ds_name)
        if verbose:
            print(f'\n=== L1 on {ds_name} (train={len(x_tr)} test={len(x_te)}) ===')

        # Fit TF-IDF once per dataset (deterministic)
        tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=20_000,
                                sublinear_tf=True, min_df=2)
        X_tr = tfidf.fit_transform(x_tr)
        X_te = tfidf.transform(x_te)

        results[ds_name] = {}
        for model_name in make_classical_models(0).keys():
            run_metrics, run_times = [], []
            for s in seeds:
                np.random.seed(s)
                est = make_classical_models(s)[model_name]
                t0 = time.time()
                est.fit(X_tr, y_tr)
                y_pred = est.predict(X_te)
                run_times.append(time.time() - t0)
                run_metrics.append(metrics(y_te, y_pred, n_cls))

            agg = {}
            for k in run_metrics[0]:
                v = np.array([r[k] for r in run_metrics], dtype=float)
                agg[f'{k}_vals'] = v.tolist()
                agg[f'{k}_mean'] = float(np.nanmean(v))
                agg[f'{k}_std']  = float(np.nanstd(v, ddof=1)) if len(v) > 1 else 0.0
            agg['time_mean'] = float(np.mean(run_times))
            agg['time_std']  = float(np.std(run_times, ddof=1))
            results[ds_name][model_name] = agg

            if verbose:
                f1m = agg['f1_macro_mean'] * 100
                f1s = agg['f1_macro_std']  * 100
                tm  = agg['time_mean']
                print(f'  {model_name:14s}  F1m={f1m:5.2f} ± {f1s:.2f}  '
                      f'time/run={tm:.3f}s')

    return results


if RUN_L1:
    L1_RESULTS = run_l1_grid(['LIAR', 'ISOT', 'FakeNewsNet'])
    # Save
    with open(WORK_DIR / 'results' / 'l1_results.json', 'w') as f:
        json.dump(L1_RESULTS, f, indent=2)
    print(f'\nL1 done. Saved to {WORK_DIR / "results" / "l1_results.json"}')
else:
    print('L1 skipped (RUN_L1=False).')


In [ ]:
# =============================================================================
# Cell 8 — Layer L1: statistical tests (Wilcoxon, Friedman, Nemenyi, Cohen's d)
# =============================================================================
from scipy.stats import wilcoxon, friedmanchisquare
import scikit_posthocs as sp


def cohen_d_from_vals(a, b):
    a = np.array(a, dtype=float); b = np.array(b, dtype=float)
    s_pooled = (((a.std(ddof=1)**2) + (b.std(ddof=1)**2)) / 2.0) ** 0.5 + 1e-12
    d = (a.mean() - b.mean()) / s_pooled
    if   abs(d) < 0.2: lab = 'negligible'
    elif abs(d) < 0.5: lab = 'small'
    elif abs(d) < 0.8: lab = 'medium'
    else:              lab = 'LARGE'
    return float(d), lab


def l1_statistics(results, alpha=0.05):
    """Run Wilcoxon vs RF, Friedman omnibus, Nemenyi posthoc, Cohen's d."""
    stats_out = {}
    for ds, models in results.items():
        stats_out[ds] = {}
        rf_vals = np.array(models['RF + TF-IDF']['f1_macro_vals'])
        for mname, agg in models.items():
            if mname == 'RF + TF-IDF':
                continue
            vals = np.array(agg['f1_macro_vals'])
            diff = vals - rf_vals
            if np.all(diff == 0):
                p = 1.0
            else:
                try:
                    _, p = wilcoxon(vals, rf_vals, alternative='two-sided')
                except ValueError:
                    p = 1.0
            d, lab = cohen_d_from_vals(vals, rf_vals)
            delta = (vals.mean() - rf_vals.mean()) * 100
            stats_out[ds][mname] = {
                'wilcoxon_p':  float(p),
                'wilcoxon_sig': bool(p < alpha),
                'cohen_d':     d,
                'cohen_d_lab': lab,
                'delta_pp':    float(delta),
            }

        # Friedman omnibus over the 4 models
        all_models = list(models.keys())
        groups = [np.array(models[m]['f1_macro_vals']) for m in all_models]
        chi2, p_friedman = friedmanchisquare(*groups)
        stats_out[ds]['_friedman'] = {
            'chi2': float(chi2), 'pval': float(p_friedman),
            'sig':  bool(p_friedman < alpha),
        }
        # Nemenyi (only if Friedman is significant)
        if p_friedman < alpha:
            matrix = np.column_stack(groups)
            nem = sp.posthoc_nemenyi_friedman(matrix)
            nem.index = all_models
            nem.columns = all_models
            stats_out[ds]['_nemenyi'] = nem.to_dict()
    return stats_out


if RUN_L1:
    L1_STATS = l1_statistics(L1_RESULTS)
    print('\n=== Wilcoxon vs RF baseline (L1) ===')
    for ds, info in L1_STATS.items():
        print(f'\n--- {ds} ---')
        if '_friedman' in info:
            fr = info['_friedman']
            tag = 'SIG' if fr['sig'] else 'NS'
            print(f'  Friedman: chi2={fr["chi2"]:.3f}, p={fr["pval"]:.4e}  [{tag}]')
        for m, s in info.items():
            if m.startswith('_'): continue
            tag = 'SIG' if s['wilcoxon_sig'] else 'NS'
            print(f'  {m:14s}  p={s["wilcoxon_p"]:.4f} [{tag}]  '
                  f'd={s["cohen_d"]:+.2f} ({s["cohen_d_lab"]})  '
                  f'Δ={s["delta_pp"]:+.2f}pp')

    with open(WORK_DIR / 'results' / 'l1_stats.json', 'w') as f:
        json.dump(L1_STATS, f, indent=2)


In [ ]:
# =============================================================================
# Cell 9 — Layer L1 figures (saved to WORK_DIR/figures/)
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if RUN_L1:
    FIG_DIR = WORK_DIR / 'figures'
    FIG_DIR.mkdir(exist_ok=True)

    datasets = list(L1_RESULTS.keys())
    models   = ['RF + TF-IDF', 'LR + TF-IDF', 'SVM + TF-IDF', 'SGD + TF-IDF']
    COLORS   = {'LR + TF-IDF': '#1f77b4', 'SVM + TF-IDF': '#2ca02c',
                'RF + TF-IDF': '#d62728', 'SGD + TF-IDF': '#ff7f0e'}

    # ---- Figure: grouped bar of macro-F1 ----
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(datasets)); w = 0.18
    for i, m in enumerate(models):
        means = [L1_RESULTS[ds][m]['f1_macro_mean']*100 for ds in datasets]
        stds  = [L1_RESULTS[ds][m]['f1_macro_std' ]*100 for ds in datasets]
        ax.bar(x + i*w - 1.5*w, means, w, yerr=stds, label=m,
               color=COLORS[m], edgecolor='black', linewidth=0.5, capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(datasets)
    ax.set_ylabel('Macro-F1 (%)'); ax.set_ylim(0, 110)
    ax.set_title(f'L1 results — {len(SEEDS)} runs per cell')
    ax.legend(loc='upper left', fontsize=9, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'l1_baselines.png', dpi=180, bbox_inches='tight')
    plt.savefig(FIG_DIR / 'l1_baselines.pdf', bbox_inches='tight')
    plt.show(); plt.close()

    # ---- Figure: LIAR boxplot ----
    if 'LIAR' in L1_RESULTS:
        fig, ax = plt.subplots(figsize=(7, 4))
        data, labels = [], []
        for m in models:
            v = np.array(L1_RESULTS['LIAR'][m]['f1_macro_vals'])*100
            data.append(v); labels.append(m.replace(' + TF-IDF',''))
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.5,
                        medianprops={'color':'black'})
        for patch, m in zip(bp['boxes'], models):
            patch.set_facecolor(COLORS[m]); patch.set_alpha(0.6)
        ax.set_ylabel('Macro-F1 (%) on LIAR')
        ax.set_title(f'Per-run distribution (n={len(SEEDS)} seeds) on LIAR')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'l1_liar_boxplot.png', dpi=180, bbox_inches='tight')
        plt.savefig(FIG_DIR / 'l1_liar_boxplot.pdf', bbox_inches='tight')
        plt.show(); plt.close()

    # ---- Figure: Cohen's d heatmap ----
    other = ['LR + TF-IDF', 'SVM + TF-IDF', 'SGD + TF-IDF']
    mat = np.zeros((len(other), len(datasets)))
    for j, ds in enumerate(datasets):
        for i, m in enumerate(other):
            mat[i,j] = L1_STATS[ds][m]['cohen_d']
    fig, ax = plt.subplots(figsize=(6, 3.5))
    im = ax.imshow(mat, cmap='RdBu_r', aspect='auto', vmin=-15, vmax=15)
    ax.set_xticks(range(len(datasets))); ax.set_xticklabels(datasets)
    ax.set_yticks(range(len(other)))
    ax.set_yticklabels([m.replace(' + TF-IDF','') for m in other])
    for i in range(len(other)):
        for j in range(len(datasets)):
            ax.text(j, i, f'{mat[i,j]:+.2f}', ha='center', va='center',
                    color='white' if abs(mat[i,j])>5 else 'black',
                    fontweight='bold', fontsize=9)
    plt.colorbar(im, ax=ax, label="Cohen's d vs RF")
    ax.set_title("L1 effect size vs RF baseline\n(|d|>=0.8 LARGE)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'l1_cohen_d.png', dpi=180, bbox_inches='tight')
    plt.savefig(FIG_DIR / 'l1_cohen_d.pdf', bbox_inches='tight')
    plt.show(); plt.close()

    print(f'\nFigures saved to {FIG_DIR}')


In [ ]:
# =============================================================================
# Cell 10 — Print paper-ready LaTeX tables (copy-paste into paper.tex)
# =============================================================================
import pandas as pd

if RUN_L1:
    # ---- Aggregate table for the paper ----
    rows = []
    for ds, models in L1_RESULTS.items():
        for m, agg in models.items():
            rows.append({
                'Dataset':    ds,
                'Model':      m,
                'Acc (%)':    f"{agg['acc_mean']*100:.2f} ± {agg['acc_std']*100:.2f}",
                'F1m (%)':    f"{agg['f1_macro_mean']*100:.2f} ± {agg['f1_macro_std']*100:.2f}",
                'P (%)':      f"{agg['prec_macro_mean']*100:.2f}",
                'R (%)':      f"{agg['rec_macro_mean']*100:.2f}",
                'MCC':        f"{agg['mcc_mean']:.3f}",
                'Time (s)':   f"{agg['time_mean']:.3f}",
            })
    df = pd.DataFrame(rows)
    print('=== L1 aggregate table (paper-ready) ===')
    print(df.to_string(index=False))

    # ---- Per-seed table on LIAR ----
    if 'LIAR' in L1_RESULTS:
        rows = []
        for m in ['RF + TF-IDF', 'LR + TF-IDF', 'SVM + TF-IDF', 'SGD + TF-IDF']:
            r = {'Model': m}
            vals = L1_RESULTS['LIAR'][m]['f1_macro_vals']
            for i, v in enumerate(vals):
                r[f's{i}'] = f'{v*100:.1f}'
            r['Mean'] = f"{L1_RESULTS['LIAR'][m]['f1_macro_mean']*100:.2f}"
            r['Std']  = f"{L1_RESULTS['LIAR'][m]['f1_macro_std']*100:.2f}"
            rows.append(r)
        df_seed = pd.DataFrame(rows)
        print('\n=== L1 per-seed F1m (%) on LIAR (paper Table 7) ===')
        print(df_seed.to_string(index=False))

    # ---- Save to CSV for paper insertion ----
    df.to_csv(WORK_DIR / 'results' / 'l1_aggregate_table.csv', index=False)
    print(f'\nCSV saved: {WORK_DIR / "results" / "l1_aggregate_table.csv"}')


## Section 3 — Layer L2: Deep-learning permutation ablation

This section builds the hybrid BERT/CNN/BiLSTM architectures and trains them on the three benchmarks, exploring all 6 orderings of {BERT, CNN, BiLSTM}.

**Compute budget per cell of the grid (NVIDIA T4 GPU):**
- BERT alone fine-tune: ~11 min/run
- Hybrid (any ordering): ~17 min/run
- 13 seeds × 6 orderings × 3 datasets = **234 runs** = ~56h total

**Resumption.** Every run saves a JSON checkpoint to `{WORK_DIR}/checkpoints/`. If the Colab session disconnects, re-run this section: completed runs are skipped automatically.

**Disabled by default.** Set `RUN_L2 = True` in Cell 2 to launch.


In [ ]:
# =============================================================================
# Cell 12 — Deep-learning architecture: 6 hybrid orderings (PyTorch)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# All 6 orderings of {BERT, CNN, BiLSTM}
ORDERINGS = {
    'O1': ('BERT', 'CNN',    'BiLSTM'),
    'O2': ('BERT', 'BiLSTM', 'CNN'),
    'O3': ('CNN',  'BERT',   'BiLSTM'),
    'O4': ('CNN',  'BiLSTM', 'BERT'),
    'O5': ('BiLSTM', 'BERT', 'CNN'),
    'O6': ('BiLSTM', 'CNN',  'BERT'),
}


class CNNBlock(nn.Module):
    """Multi-kernel CNN with kernels k ∈ {2,3,4}, 128 maps each."""
    def __init__(self, in_dim, kernel_sizes=(2,3,4), n_maps=128, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_dim, n_maps, kernel_size=k, padding=k//2)
            for k in kernel_sizes
        ])
        self.out_dim = n_maps * len(kernel_sizes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):                   # x: (B, T, D)
        x = x.transpose(1, 2)               # -> (B, D, T)
        outs = [F.relu(c(x)) for c in self.convs]
        # pad to common length
        min_t = min(o.size(-1) for o in outs)
        outs = [o[..., :min_t] for o in outs]
        x = torch.cat(outs, dim=1)          # (B, sum_maps, T)
        return self.dropout(x.transpose(1, 2))   # (B, T, sum_maps)


class BiLSTMBlock(nn.Module):
    """BiLSTM(h=256) + scaled dot-product self-attention."""
    def __init__(self, in_dim, hidden=256, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True, bidirectional=True)
        self.q = nn.Linear(2*hidden, 2*hidden)
        self.k = nn.Linear(2*hidden, 2*hidden)
        self.v = nn.Linear(2*hidden, 2*hidden)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = 2*hidden

    def forward(self, x):                   # (B, T, D)
        h, _ = self.lstm(x)                 # (B, T, 2H)
        d_k = h.size(-1)
        Q, K, V = self.q(h), self.k(h), self.v(h)
        attn = F.softmax((Q @ K.transpose(-2,-1)) / (d_k**0.5), dim=-1)
        return self.dropout(attn @ V)       # (B, T, 2H)


class HybridFND(nn.Module):
    """Hybrid model with permutable {BERT, CNN, BiLSTM} zone and fixed tail."""

    def __init__(self, ordering, n_classes, use_gnn=False,
                 bert_model='bert-base-uncased'):
        super().__init__()
        from transformers import AutoModel
        self.ordering = ordering
        self.bert = AutoModel.from_pretrained(bert_model)
        bert_dim = self.bert.config.hidden_size            # 768

        # Build the permutable stack: each stage receives `cur_dim` from prev
        self.stages = nn.ModuleList()
        self.stage_names = []
        cur_dim = None        # initialised when first stage runs

        for name in ordering:
            if name == 'BERT':
                # BERT is the encoder; runs whenever it appears, on tokens at first
                # position, or on a projected representation otherwise.
                self.stage_names.append('BERT')
                if cur_dim is None:
                    cur_dim = bert_dim
                else:
                    # need projection back to BERT-compatible dim
                    self.stages.append(nn.Linear(cur_dim, bert_dim))
                    cur_dim = bert_dim
                # placeholder (BERT itself called in forward)
                self.stages.append(nn.Identity())
            elif name == 'CNN':
                self.stage_names.append('CNN')
                in_dim = bert_dim if cur_dim is None else cur_dim
                blk = CNNBlock(in_dim)
                self.stages.append(blk); cur_dim = blk.out_dim
            elif name == 'BiLSTM':
                self.stage_names.append('BiLSTM')
                in_dim = bert_dim if cur_dim is None else cur_dim
                blk = BiLSTMBlock(in_dim)
                self.stages.append(blk); cur_dim = blk.out_dim

        # Fixed tail
        self.self_attn = nn.MultiheadAttention(embed_dim=cur_dim, num_heads=4,
                                               dropout=0.3, batch_first=True)
        self.layer_norm = nn.LayerNorm(cur_dim)
        self.use_gnn = use_gnn
        if use_gnn:
            self.gnn_proj1 = nn.Linear(cur_dim, 256)
            self.gnn_proj2 = nn.Linear(256, 256)
            self.classifier = nn.Linear(256, n_classes)
        else:
            self.classifier = nn.Linear(cur_dim, n_classes)

    def forward(self, input_ids, attention_mask=None, edge_index=None):
        # Run permutable stages in order
        x = None
        stage_idx = 0
        for name in self.ordering:
            if name == 'BERT':
                if x is None:
                    # First stage: feed token IDs directly
                    out = self.bert(input_ids=input_ids,
                                    attention_mask=attention_mask)
                    x = out.last_hidden_state    # (B, T, 768)
                    stage_idx += 1               # skip Identity placeholder
                else:
                    # Need a projection layer then BERT (rare path)
                    x = self.stages[stage_idx](x)        # Linear proj
                    stage_idx += 1
                    # Approximate BERT by running its encoder on the projection
                    out = self.bert.encoder(x)
                    x = out.last_hidden_state if hasattr(out,'last_hidden_state') else x
                    stage_idx += 1
            else:
                x = self.stages[stage_idx](x)
                stage_idx += 1

        # Self-attention residual (fixed tail)
        res, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.layer_norm(x + res)

        # Global pool
        pooled = x.mean(dim=1)                      # (B, D)

        # GNN (optional, only on FakeNewsNet)
        if self.use_gnn:
            pooled = F.relu(self.gnn_proj1(pooled))
            pooled = F.relu(self.gnn_proj2(pooled))

        return self.classifier(pooled)


# Smoke test
print('Architecture defined for orderings:', list(ORDERINGS.keys()))
print('HybridFND module class:', HybridFND.__name__)


In [ ]:
# =============================================================================
# Cell 13 — Layer L2 training loop (with checkpointing for session resumption)
# =============================================================================
import time, json, hashlib
from pathlib import Path


def ckpt_key(ordering, dataset, seed, use_gnn=False):
    tag = f'GNN' if use_gnn else 'baseline'
    return f'{dataset}__{ordering}__{tag}__seed{seed:02d}'


def load_ckpt(key):
    p = WORK_DIR / 'checkpoints' / f'{key}.json'
    if p.exists():
        return json.loads(p.read_text())
    return None


def save_ckpt(key, payload):
    p = WORK_DIR / 'checkpoints' / f'{key}.json'
    p.write_text(json.dumps(payload, indent=2))


def train_one_run(ordering, dataset_name, seed, use_gnn=False,
                  epochs=3, batch_size=16, max_len=128, lr=2e-5):
    """Train one hybrid model on one dataset with one seed.

    Returns dict with metrics + wall-clock time.
    """
    from transformers import AutoTokenizer
    from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

    # Load data
    x_tr, y_tr, x_te, y_te, n_cls = load_dataset(dataset_name)

    # Tokenise
    tok = AutoTokenizer.from_pretrained('bert-base-uncased')
    enc_tr = tok(x_tr, padding='max_length', truncation=True,
                 max_length=max_len, return_tensors='pt')
    enc_te = tok(x_te, padding='max_length', truncation=True,
                 max_length=max_len, return_tensors='pt')
    y_tr_t = torch.tensor(y_tr, dtype=torch.long)
    y_te_t = torch.tensor(y_te, dtype=torch.long)

    # Determinism
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = HybridFND(ORDERINGS[ordering], n_classes=n_cls,
                      use_gnn=use_gnn).to(device)

    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()

    # Train
    t0 = time.time()
    model.train()
    n = enc_tr['input_ids'].size(0)
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            ids   = enc_tr['input_ids'][idx].to(device)
            mask  = enc_tr['attention_mask'][idx].to(device)
            y_b   = y_tr_t[idx].to(device)
            logits = model(ids, attention_mask=mask)
            loss = crit(logits, y_b)
            optim.zero_grad(); loss.backward(); optim.step()

    # Evaluate
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, enc_te['input_ids'].size(0), batch_size):
            ids   = enc_te['input_ids'][i:i+batch_size].to(device)
            mask  = enc_te['attention_mask'][i:i+batch_size].to(device)
            logits = model(ids, attention_mask=mask)
            preds.append(logits.argmax(dim=-1).cpu().numpy())
    y_pred = np.concatenate(preds)
    dt = time.time() - t0

    return {
        'ordering':    ordering, 'dataset': dataset_name, 'seed': seed,
        'use_gnn':     use_gnn,
        'acc':         float(accuracy_score(y_te, y_pred)),
        'f1_macro':    float(f1_score(y_te, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_te, y_pred, average='weighted', zero_division=0)),
        'mcc':         float(matthews_corrcoef(y_te, y_pred)),
        'time_s':      float(dt),
    }


def run_l2_grid(orderings=ORDERINGS_L2, datasets=DATASETS_L2,
                seeds=SEEDS, use_gnn=False, verbose=True):
    """Run the full L2 grid with checkpointing."""
    results = []
    n_total = len(orderings) * len(datasets) * len(seeds)
    n_done = 0

    for ordering in orderings:
        for ds_name in datasets:
            for seed in seeds:
                key = ckpt_key(ordering, ds_name, seed, use_gnn)
                cached = load_ckpt(key)
                if cached is not None:
                    if verbose:
                        print(f'[cache] {key}')
                    results.append(cached); n_done += 1; continue

                if verbose:
                    print(f'[train] {key}  ({n_done+1}/{n_total})')
                r = train_one_run(ordering, ds_name, seed, use_gnn=use_gnn)
                save_ckpt(key, r)
                results.append(r); n_done += 1

                if verbose:
                    print(f'   F1m={r["f1_macro"]*100:.2f}  '
                          f'acc={r["acc"]*100:.2f}  time={r["time_s"]:.1f}s')

    return results


if RUN_L2:
    L2_RUNS = run_l2_grid()
    print(f'\nL2 done: {len(L2_RUNS)} runs collected.')
    # Save flat results
    with open(WORK_DIR / 'results' / 'l2_runs.json', 'w') as f:
        json.dump(L2_RUNS, f, indent=2)
else:
    print('L2 skipped (RUN_L2=False). Enable in Cell 2 to run on GPU.')
    L2_RUNS = []


In [ ]:
# =============================================================================
# Cell 14 — L2: aggregate per-(ordering, dataset) and run statistical tests
# =============================================================================

def aggregate_l2(runs):
    """Aggregate flat runs into (ordering, dataset) -> stats dict."""
    from collections import defaultdict
    grouped = defaultdict(list)
    for r in runs:
        grouped[(r['ordering'], r['dataset'])].append(r)

    agg = {}
    for (ord_, ds), rs in grouped.items():
        f1m = np.array([r['f1_macro'] for r in rs])
        acc = np.array([r['acc'] for r in rs])
        mcc = np.array([r['mcc'] for r in rs])
        tms = np.array([r['time_s'] for r in rs])
        agg[(ord_, ds)] = {
            'n_runs':    len(rs),
            'f1m_mean':  float(f1m.mean()), 'f1m_std': float(f1m.std(ddof=1)),
            'f1m_vals':  f1m.tolist(),
            'acc_mean':  float(acc.mean()), 'acc_std': float(acc.std(ddof=1)),
            'mcc_mean':  float(mcc.mean()),
            'time_mean': float(tms.mean()),
        }
    return agg


def l2_statistics(agg, base_order='O1'):
    """Wilcoxon vs the best ordering O1, on each dataset."""
    out = {}
    datasets = sorted({d for (_,d) in agg.keys()})
    for ds in datasets:
        if (base_order, ds) not in agg:
            continue
        base_vals = np.array(agg[(base_order, ds)]['f1m_vals'])
        out[ds] = {}
        for (ord_, ds2) in agg:
            if ds2 != ds or ord_ == base_order:
                continue
            vals = np.array(agg[(ord_, ds2)]['f1m_vals'])
            if len(vals) != len(base_vals) or np.all(vals == base_vals):
                p = 1.0
            else:
                try:
                    _, p = wilcoxon(vals, base_vals, alternative='two-sided')
                except ValueError:
                    p = 1.0
            d, lab = cohen_d_from_vals(base_vals, vals)
            delta_pp = (base_vals.mean() - vals.mean()) * 100
            out[ds][ord_] = {
                'wilcoxon_p_vs_O1': float(p),
                'cohen_d':          d,
                'cohen_d_lab':      lab,
                'delta_pp_vs_O1':   float(delta_pp),
            }
    return out


if RUN_L2 and L2_RUNS:
    L2_AGG   = aggregate_l2(L2_RUNS)
    L2_STATS = l2_statistics(L2_AGG)

    print('\n=== L2 ordering ablation results ===')
    print(f'{"Ordering":4s}  {"Dataset":12s}  {"F1m":>10s}  {"Acc":>10s}  {"n":>4s}')
    for (ord_, ds), s in sorted(L2_AGG.items()):
        print(f'  {ord_:4s}  {ds:12s}  '
              f'{s["f1m_mean"]*100:5.2f}±{s["f1m_std"]*100:.2f}  '
              f'{s["acc_mean"]*100:5.2f}±{s["acc_std"]*100:.2f}  '
              f'{s["n_runs"]:4d}')

    print('\n=== Wilcoxon vs O1 ===')
    for ds, info in L2_STATS.items():
        print(f'\n--- {ds} ---')
        for ord_, s in sorted(info.items()):
            sig = '**' if s['wilcoxon_p_vs_O1']<0.01 else ('*' if s['wilcoxon_p_vs_O1']<0.05 else 'NS')
            print(f'  O1 vs {ord_}: p={s["wilcoxon_p_vs_O1"]:.4f} [{sig}]  '
                  f'd={s["cohen_d"]:+.2f}  Δ={s["delta_pp_vs_O1"]:+.2f}pp')

    # Save
    out = {'agg': {f'{o}|{d}': v for (o,d),v in L2_AGG.items()},
           'stats': L2_STATS}
    with open(WORK_DIR / 'results' / 'l2_summary.json', 'w') as f:
        json.dump(out, f, indent=2)
else:
    print('L2 results unavailable (RUN_L2=False or empty).')


## Section 4 — Layer L3: GNN ablation on FakeNewsNet

The GNN augmentation is only meaningful when social propagation graphs are available — i.e., on FakeNewsNet. This section trains O1 with and without the GNN tail (13 seeds each) and runs the Wilcoxon test. The expected result is +3.8 pp macro-F1 (p < 0.001, large effect).

This is the smallest experiment in the notebook: 2 configurations × 13 seeds = **26 runs on GPU**, ~7 hours total.


In [ ]:
# =============================================================================
# Cell 16 — Layer L3: GNN ablation on FakeNewsNet (O1 vs O1+GNN)
# =============================================================================

def run_l3_gnn_ablation(seeds=SEEDS, verbose=True):
    """Train O1 with and without GNN tail on FakeNewsNet."""
    runs = []
    for use_gnn in (False, True):
        tag = 'O1+GNN' if use_gnn else 'O1'
        if verbose:
            print(f'\n=== L3: {tag} on FakeNewsNet ===')
        for s in seeds:
            key = ckpt_key('O1', 'FakeNewsNet', s, use_gnn=use_gnn)
            cached = load_ckpt(key)
            if cached is not None:
                if verbose: print(f'  [cache] seed {s}')
                runs.append(cached); continue
            r = train_one_run('O1', 'FakeNewsNet', s, use_gnn=use_gnn)
            save_ckpt(key, r); runs.append(r)
            if verbose:
                print(f'  seed {s}: F1m={r["f1_macro"]*100:.2f}  '
                      f'time={r["time_s"]:.1f}s')
    return runs


if RUN_L3:
    L3_RUNS = run_l3_gnn_ablation()
    # Aggregate
    no_gnn = np.array([r['f1_macro'] for r in L3_RUNS if not r['use_gnn']])
    yes_gnn = np.array([r['f1_macro'] for r in L3_RUNS if      r['use_gnn']])
    delta = (yes_gnn.mean() - no_gnn.mean()) * 100
    try:
        _, p = wilcoxon(yes_gnn, no_gnn, alternative='two-sided')
    except ValueError:
        p = 1.0
    d, lab = cohen_d_from_vals(yes_gnn, no_gnn)
    print('\n=== L3 GNN ablation summary ===')
    print(f'O1 no GNN:   {no_gnn.mean()*100:.2f} ± {no_gnn.std(ddof=1)*100:.2f}')
    print(f'O1 + GNN:    {yes_gnn.mean()*100:.2f} ± {yes_gnn.std(ddof=1)*100:.2f}')
    print(f'Δ = {delta:+.2f} pp;  Wilcoxon p={p:.4f}; '
          f'Cohen d={d:+.2f} ({lab})')

    with open(WORK_DIR / 'results' / 'l3_summary.json', 'w') as f:
        json.dump({
            'no_gnn_mean': float(no_gnn.mean()), 'no_gnn_std': float(no_gnn.std(ddof=1)),
            'gnn_mean':    float(yes_gnn.mean()), 'gnn_std':    float(yes_gnn.std(ddof=1)),
            'delta_pp':    float(delta),
            'wilcoxon_p':  float(p),
            'cohen_d':     d, 'cohen_d_lab': lab,
        }, f, indent=2)
else:
    print('L3 skipped (RUN_L3=False). Enable in Cell 2 to run on GPU.')


## Section 5 — Summary

This notebook has reproduced the three layers of experiments described in the paper:

| Layer | Runs | Hardware | Wall-clock | Status |
|---|---|---|---|---|
| L1 (classical) | 4 models × 3 ds × 13 seeds = 156 | CPU | ~1 min | always runs |
| L2 (ordering) | 6 orderings × 3 ds × 13 seeds = 234 | GPU (T4) | ~56h with checkpointing | optional |
| L3 (GNN) | 2 × 13 = 26 | GPU (T4) | ~7h | optional |

**All artefacts produced:**
- `results/l1_results.json`, `l1_stats.json`, `l1_aggregate_table.csv`
- `results/l2_runs.json`, `l2_summary.json`
- `results/l3_summary.json`
- `figures/l1_baselines.{png,pdf}`, `l1_liar_boxplot.{png,pdf}`, `l1_cohen_d.{png,pdf}`
- `checkpoints/{Dataset}__{Ordering}__{tag}__seed{NN}.json` (one per run, for resumption)

**To export results to the paper:**
1. Copy the printed mean ± std numbers from Cell 10 into the `\multirow` blocks of Table 8 in `paper.tex`.
2. Copy the per-seed numbers from Cell 10 into Table 7.
3. After running L2 (Cell 13), copy the ordering ablation numbers into Table 11.
4. After running L3 (Cell 16), copy the GNN ablation numbers into Table 12.

The paper compiles with `pdflatex paper.tex` (twice for cross-refs).
